# Displacement of an object from ground-plane homography

**Case:** an object on the ground (a Traffic Delineator Post, TDP) filmed by an oblique-view CCTV camera; its
displacement is estimated in georeferenced metres, with uncertainty.

**Approach of this notebook:**
- **Per-frame homography**: in every frame the ground control points (GCP) are digitized and
  their own homography is computed -> the result is immune to camera motion/vibration. The
  reference frame is the ground plane.
- **Scale anchoring with a known object** (e.g. a car's wheelbase in the scene): fixes the
  scale to ~cm precision, which is the component that matters most for displacement and the
  one most affected by satellite-reference error.
- **Monte Carlo uncertainty**: perturbs clicks and references to produce bands (p2.5-p97.5,
  p25-p75) on position.

**Execution order:** run the cells in order. Step 1 (parameters/functions) is required; step 2
(frame explorer) is optional. Digitization (7) and review (8) are interactive; the rest is
automatic. Everything digitized is saved under the `<video>_picks/` folder and is resumable if
the kernel restarts.

**Requirements:** `opencv-python`, `numpy`, `matplotlib`, `ipympl`, `ipywidgets`, `geopandas`.
If a figure renders static, install `ipympl` and restart the kernel.

**Scope of this cleaned version:** this notebook only produces the per-target displacement
trace (position + Monte Carlo covariance) that feeds the `fusion_BLUE` notebook. The original
working version also had optional sections for velocity, directional PGD/PGV peaks and an
annotated trajectory video; those are not needed for the paper and were removed here.

## 1. Configuration (required)

Run in order: imports -> parameters -> functions/classes -> derived values. If you change a
parameter, re-run from the parameters cell onward.

In [ ]:
%matplotlib widget
# Interactive backend inside the notebook. Do NOT use %matplotlib qt (it clashes with OpenCV's
# DLLs on Windows and crashes the kernel). Requires ipympl:  pip install ipympl
import os, json, csv
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.patches import Ellipse

print('OpenCV', cv2.__version__, '| NumPy', np.__version__, '| backend', plt.get_backend())


In [ ]:
# ============================  PARAMETERS  ============================
# EDIT ONLY THIS CELL for your own case. The rest of the notebook uses these values.

# EDIT THIS: base working folder (videos and GCP files live here).
BASE_PATH   = r"\path_to\tdp-displacement-cctv"
VIEW_NAME   = "Vista_2"     # view name (used to save results)
TARGET      = "R"           # target name -- this project only tracks the right TDP (R), in both views
N_TARGETS   = 1             # number of targets tracked per frame -- always 1 in this project

VIDEO_PATH             = os.path.join(BASE_PATH, f"{VIEW_NAME}.mp4")                     # CCTV video
REF_POINTS_PATH_QGIS   = os.path.join(BASE_PATH, "ref_pionts_v4.gpkg")                   # GCP in lat/long (.gpkg)
REF_POINTS_PATH_INSITU = os.path.join(BASE_PATH, "ref_points_dxf.txt")                   # GCP in local (x,y) coords (m)
OUTPUT_PATH             = os.path.join(BASE_PATH, "output", f"{VIEW_NAME}_{TARGET}")     # results folder
CLICK_PATH              = os.path.join(BASE_PATH, f"{VIEW_NAME}_picks", f"Target_{TARGET}")  # digitization-click folder
os.makedirs(OUTPUT_PATH, exist_ok=True)

if VIEW_NAME == "Vista_1":
    FRAME_REF   = 2100      # reference frame to digitize GCP, anchor scale and verify
    FRAME_START = 2243      # start of the analysis window (ideally slightly before the event)
    FRAME_END   = 2404      # end (ideally once the object is at rest)
    N_GCP       = 6         # number of GCP to digitize (minimum 3, ideally 4+)

elif VIEW_NAME == "Vista_2":
    FRAME_REF   = 76        # reference frame to digitize GCP, anchor scale and verify
    FRAME_START = 536       # start of the analysis window (ideally slightly before the event)
    FRAME_END   = 736       # end (ideally once the object is at rest)
    N_GCP       = 7         # number of GCP to digitize (minimum 3, ideally 4+)

T_STEP_S    = 0.1      # temporal sampling step (s)

# --- Uncertainty (Monte Carlo) ---
SIGMA_CLICK_PX    = 1.0    # digitization error in pixels (with HD video and zoom, ~1)
SIGMA_GCP_WORLD_M = 0.01   # residual RELATIVE error of the GCP AFTER anchoring the scale (m).
                           #   Before anchoring (if you skip section 5): use 0.3-0.6.
N_SIM             = 300    # number of Monte Carlo simulations

print('Parameters loaded. Remember to re-run this cell if you change anything.')


In [ ]:
# --- Tidy figure-saving utility (for the report) ---
# Every important figure is saved to OUTPUT_PATH/figures/{fig_id}.png and registered in
# OUTPUT_PATH/figures_manifest.json (view, target, path, caption) so it can later be inserted
# automatically into the Word report with insert_figures_into_report.py.
FIGURES_DIR = os.path.join(OUTPUT_PATH, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
_FIG_MANIFEST_PATH = os.path.join(OUTPUT_PATH, 'figures_manifest.json')

def save_fig(fig, fig_id, caption, dpi=300):
    """Save `fig` to FIGURES_DIR/{fig_id}.png and add/update its entry in this view/target's
    JSON manifest (does not duplicate entries if you re-run the cell)."""
    path = os.path.join(FIGURES_DIR, f'{fig_id}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    entry = {'fig_id': fig_id, 'path': path, 'caption': caption,
             'view': VIEW_NAME, 'target': TARGET}
    manifest = []
    if os.path.exists(_FIG_MANIFEST_PATH):
        with open(_FIG_MANIFEST_PATH, encoding='utf-8') as f:
            manifest = json.load(f)
    manifest = [m for m in manifest if m['fig_id'] != fig_id] + [entry]
    with open(_FIG_MANIFEST_PATH, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=1, ensure_ascii=False)
    print(f'Figure saved: {path}')


In [ ]:
# --- Base functions + keymap cleanup (run once) ---
_UNDIST_MAPS = None  # (map1, map2) once distortion has been calibrated (section 1b); None = uncorrected
def get_frame(video_path, idx):
    """Frame `idx` of the video as an RGB array. If a distortion calibration is active
    (_UNDIST_MAPS), it is returned ALREADY corrected (undistorted)."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise ValueError(f'Could not read frame {idx}.')
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    if _UNDIST_MAPS is not None:
        frame = cv2.remap(frame, _UNDIST_MAPS[0], _UNDIST_MAPS[1], cv2.INTER_LINEAR)
    return frame

def compute_homography(img_pts, world_pts):
    """Image->world homography by least squares (>=4 non-collinear correspondences)."""
    H, _ = cv2.findHomography(np.array(img_pts, np.float32),
                              np.array(world_pts, np.float32), method=0)
    if H is None:
        raise RuntimeError('Could not compute the homography (collinear points?).')
    return H

def img_to_world(H, pt):
    """Project an image point (u,v) to world coordinates (X,Y)."""
    return cv2.perspectiveTransform(np.array([[pt]], np.float32), H)[0, 0]

HANDLE_TOL_PX = 12  # radius (screen px) to "grab" handles (template / guide lines)

# Release matplotlib's default keymaps that clash with the picker's own shortcuts
# ('l' = log y-scale used to make the image "disappear"; arrows = view navigation).
for _km in [k for k in plt.rcParams if k.startswith('keymap.')]:
    for _key in ('l', 'k', 'L', 'a', 't', 'left', 'right', 'up', 'down'):
        while _key in plt.rcParams[_km]:
            plt.rcParams[_km].remove(_key)

_NUDGE = {'left': (-1, 0), 'right': (1, 0), 'up': (0, -1), 'down': (0, 1)}  # image y grows downward

TARGET_COLORS = ['red', 'blue', 'purple', 'darkorange']
TMPL_COLORS = ['yellow', 'magenta', 'orange', 'white']
TARGET_BGR = [(0, 0, 255), (255, 0, 0), (128, 0, 128), (0, 140, 255)]


In [ ]:
class PointPicker:
    """Captures N clicks on an image without blocking the kernel (avoids the ginput+ipympl hang).
    Usage:
        picker = PointPicker(frame_ref, n=4, title='...')   # the figure appears
        # click n times directly on the displayed figure
        pts = picker.points                                 # in the NEXT cell, once clicks are done

    Key `z`: undoes the LAST clicked point (even after completing all N clicks).
    Guide lines (optional, visual aid only): key `l` adds a draggable dotted line
    (endpoints = squares, to orient/lengthen; centre circle = translate),
    key `d` removes the last one. They are not recorded as data and do not affect the
    captured points. The toolbar's zoom/pan does not register clicks while active.
    """
    def __init__(self, img, n, title, figsize=(9, 7.5)):
        self.n = n
        self.points = []
        self._point_artists = []
        self.title_base = title
        self.ref_lines = []
        self._drag = None
        self._connected = True
        self.fig, self.ax = plt.subplots(figsize=figsize)
        self.ax.imshow(img)
        self.ax.axis('off')
        self.ax.set_title(f'{title} -- 0/{n} clicks')
        self.cid = self.fig.canvas.mpl_connect('button_press_event', self._onclick)
        self.fig.canvas.mpl_connect('motion_notify_event', self._onmotion)
        self.fig.canvas.mpl_connect('button_release_event', self._onrelease)
        self.fig.canvas.mpl_connect('key_press_event', self._onkey)
        plt.show()

    def _register_point(self, x, y):
        self.points.append((x, y))
        marker, = self.ax.plot(x, y, 'r+', markersize=14, mew=2)
        label = self.ax.annotate(str(len(self.points)), (x, y),
                                  color='yellow', fontsize=10, xytext=(5, 5), textcoords='offset points')
        self._point_artists.append((marker, label))
        if len(self.points) >= self.n:
            self.ax.set_title(f'{self.title_base} -- complete ({self.n}/{self.n})')
            self.fig.canvas.mpl_disconnect(self.cid)
            self._connected = False
        else:
            self.ax.set_title(f'{self.title_base} -- {len(self.points)}/{self.n} clicks')
        self.fig.canvas.draw_idle()

    def _undo_point(self):
        if not self.points:
            return
        self.points.pop()
        marker, label = self._point_artists.pop()
        marker.remove()
        label.remove()
        if not self._connected:
            self.cid = self.fig.canvas.mpl_connect('button_press_event', self._onclick)
            self._connected = True
        self.ax.set_title(f'{self.title_base} -- {len(self.points)}/{self.n} clicks')
        self.fig.canvas.draw_idle()

    def _onclick(self, event):
        if event.xdata is None or event.ydata is None:
            return  # click outside the axes
        if getattr(self.fig.canvas.toolbar, 'mode', ''):
            return  # zoom/pan active: this click is for navigation, not data
        hit = self._hit_handle(event)
        if hit is not None:
            rl, which = hit
            (x0,), (y0,) = rl['h0'].get_data()
            (x1,), (y1,) = rl['h1'].get_data()
            self._drag = (rl, which, (event.xdata, event.ydata), ((x0, y0), (x1, y1)))
            return
        if len(self.points) >= self.n:
            return
        self._register_point(event.xdata, event.ydata)

    def reset(self):
        for marker, label in self._point_artists:
            marker.remove()
            label.remove()
        self.points = []
        self._point_artists = []
        self.ax.set_title(f'{self.title_base} -- 0/{self.n} clicks')
        if not self._connected:
            self.cid = self.fig.canvas.mpl_connect('button_press_event', self._onclick)
            self._connected = True
        self.fig.canvas.draw_idle()

    # --- guide lines (visual aid only, not recorded as data) ---
    def _add_refline(self):
        x0, x1 = self.ax.get_xlim()
        y0, y1 = self.ax.get_ylim()
        xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
        dx = (x1 - x0) * 0.2
        line, = self.ax.plot([xm - dx, xm + dx], [ym, ym], '--', color='cyan', lw=1.5, zorder=4)
        h0, = self.ax.plot([xm - dx], [ym], 's', color='cyan', ms=7, zorder=5)
        h1, = self.ax.plot([xm + dx], [ym], 's', color='cyan', ms=7, zorder=5)
        hm, = self.ax.plot([xm], [ym], 'o', mec='cyan', mfc='none', ms=7, mew=1.5, zorder=5)
        self.ref_lines.append({'line': line, 'h0': h0, 'h1': h1, 'hm': hm})
        self.fig.canvas.draw_idle()

    def _remove_last_refline(self):
        if not self.ref_lines:
            return
        rl = self.ref_lines.pop()
        for a in rl.values():
            a.remove()
        self.fig.canvas.draw_idle()

    def _sync_refline(self, rl):
        (x0,), (y0,) = rl['h0'].get_data()
        (x1,), (y1,) = rl['h1'].get_data()
        rl['line'].set_data([x0, x1], [y0, y1])
        rl['hm'].set_data([(x0 + x1) / 2], [(y0 + y1) / 2])

    def _hit_handle(self, event):
        if event.x is None or event.y is None:
            return None
        for rl in reversed(self.ref_lines):
            for which in ('h0', 'h1', 'hm'):
                (px,), (py,) = rl[which].get_data()
                X, Y = self.ax.transData.transform((px, py))
                if (X - event.x) ** 2 + (Y - event.y) ** 2 <= HANDLE_TOL_PX ** 2:
                    return rl, which
        return None

    def _onmotion(self, event):
        if self._drag is None or event.xdata is None or event.ydata is None:
            return
        rl, which, start_xy, init = self._drag
        if which == 'hm':
            ddx = event.xdata - start_xy[0]
            ddy = event.ydata - start_xy[1]
            (ix0, iy0), (ix1, iy1) = init
            rl['h0'].set_data([ix0 + ddx], [iy0 + ddy])
            rl['h1'].set_data([ix1 + ddx], [iy1 + ddy])
        else:
            rl[which].set_data([event.xdata], [event.ydata])
        self._sync_refline(rl)
        self.fig.canvas.draw_idle()

    def _onrelease(self, event):
        self._drag = None

    def _onkey(self, event):
        if event.key == 'l':
            self._add_refline()
        elif event.key == 'd':
            self._remove_last_refline()
        elif event.key == 'z':
            self._undo_point()


In [ ]:
class StraightLinePicker:
    """Digitizes points along several lines that are straight in the WORLD.
    click = add point to the current line | n = finish the line and start another | z = undo.
    Click well-spread points along each line, especially toward the frame EDGES (where
    distortion is largest). Needs >=2 lines with >=3 points each (more is better)."""
    def __init__(self, img, title='Straight lines for calibration', figsize=(10, 7.5)):
        self.lines = []
        self.current = []
        self._artists = []
        self.title = title
        self.fig, self.ax = plt.subplots(figsize=figsize)
        self.ax.imshow(img); self.ax.axis('off')
        self.fig.canvas.mpl_connect('button_press_event', self._onclick)
        self.fig.canvas.mpl_connect('key_press_event', self._onkey)
        self._redraw()
        plt.show()

    def _redraw(self):
        for a in self._artists:
            a.remove()
        self._artists = []
        cols = plt.cm.tab10.colors
        for i, ln in enumerate(self.lines + [self.current]):
            if not ln:
                continue
            xs = [p[0] for p in ln]; ys = [p[1] for p in ln]
            self._artists += self.ax.plot(xs, ys, '-o', color=cols[i % 10], ms=4, lw=1)
        self.ax.set_title(f'{self.title} -- line {len(self.lines)+1}, {len(self.current)} pts  '
                          f'|  click=point, n=new line, z=undo')
        self.fig.canvas.draw_idle()

    def _onclick(self, e):
        if e.xdata is None or getattr(self.fig.canvas.toolbar, 'mode', ''):
            return
        self.current.append((float(e.xdata), float(e.ydata)))
        self._redraw()

    def _onkey(self, e):
        if e.key == 'n' and len(self.current) >= 2:
            self.lines.append(self.current); self.current = []; self._redraw()
        elif e.key == 'z':
            if self.current:
                self.current.pop()
            elif self.lines:
                self.current = self.lines.pop()
            self._redraw()

    def get_lines(self):
        out = [l for l in self.lines if len(l) >= 3]
        if len(self.current) >= 3:
            out = out + [self.current]
        return out


In [ ]:
class SequencePicker:
    """Click-through over a sequence of frames: in each one, the `n_refs` GCP (always in the
    same order) and then `n_targets` targets to track (e.g. 2 TDPs). The title indicates
    which point is next. Once a frame's points are complete, it advances on its own.

    Keys:
      n = skip the whole frame | z = undo the last click (or go back to the previous frame)
      l = add a guide line     | d = remove the last guide line
      t = template of the target CURRENTLY being clicked: starts the OUTLINE; t again switches to ANCHOR
      a = accept the template's anchor as the point for the current target (advances on its own)
      arrows = move the current target's template by 1 px (shift+arrow = 5 px)

    Templates (one per target, distinct colours): `t` starts drawing the OUTLINE of the
    target currently being clicked (each click adds a vertex); `t` again switches to ANCHOR
    mode, where the next click sets the BASE POINT (the value that gets recorded) and closes
    that template. Each template persists across frames: drag it by its handle (or nudge with
    arrow keys) until it lines up, then accept with `a`. Saved to templates.json (auto-migrates
    from a single-template template.json if one existed).

    Clicks made while zoom/pan is active are not registered. Immediate persistence to
    save_dir: frame_XXXXXX.jpg (annotated) + clicks.json (resumable). If clicks.json holds
    records from a format with fewer targets, they are migrated by filling the new ones with a
    default point (20, 20) that you should reposition (a message reports how many frames were
    affected).
    """
    def __init__(self, video_path, frame_idxs, fps, n_refs, n_targets=1,
                 save_dir=None, resume=True, figsize=(9, 7)):
        self.video_path = video_path
        self.idxs = list(frame_idxs)
        self.fps = fps
        self.n_refs = n_refs
        self.n_targets = n_targets
        self.n_total = n_refs + n_targets
        self.k = 0
        self.current = []
        self.records = {}
        self.ref_lines = []
        self._drag = None
        self.templates = [None] * n_targets
        self.tmpl_def_mode = False       # False | 'outline' | 'anchor'
        self.tmpl_def_target = None
        self.tmpl_def_pts = []
        self.save_dir = save_dir
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            self.json_path = os.path.join(save_dir, 'clicks.json')
            self.tmpl_path = os.path.join(save_dir, 'templates.json')
            _legacy_tmpl_path = os.path.join(save_dir, 'template.json')
            if resume and os.path.exists(self.json_path):
                with open(self.json_path, encoding='utf-8') as f:
                    prev = {int(k): v for k, v in json.load(f).items()}
                recs, migrated = {}, 0
                for idx in self.idxs:
                    if idx not in prev:
                        continue
                    v = prev[idx]
                    if v is None:
                        recs[idx] = None
                    elif len(v) == self.n_total:
                        recs[idx] = v
                    elif len(v) == self.n_refs + 1 and self.n_targets > 1:
                        recs[idx] = list(v) + [[20.0, 20.0] for _ in range(self.n_targets - 1)]
                        migrated += 1
                    # other lengths: discarded (incompatible format)
                self.records = recs
                while self.k < len(self.idxs) and self.idxs[self.k] in self.records:
                    self.k += 1
                if self.records:
                    print(f'Progress recovered: {len(self.records)} frames already resolved, '
                          f'continuing at {min(self.k + 1, len(self.idxs))}/{len(self.idxs)}.')
                if migrated:
                    print(f'(!) {migrated} frame(s) migrated to the new format ({self.n_targets} '
                          f'targets): the new target(s) were set to (20, 20) by default -- '
                          f'review and reposition them (Reviewer: key "a"/"b" after '
                          f'defining its template).')
            if resume and os.path.exists(self.tmpl_path):
                with open(self.tmpl_path, encoding='utf-8') as f:
                    loaded = json.load(f)
                for t in range(min(len(loaded), self.n_targets)):
                    self.templates[t] = loaded[t]
                n_ok = sum(1 for t in self.templates if t)
                if n_ok:
                    print(f'{n_ok} template(s) recovered from templates.json.')
            elif resume and os.path.exists(_legacy_tmpl_path):
                with open(_legacy_tmpl_path, encoding='utf-8') as f:
                    self.templates[0] = json.load(f)
                self._tmpl_save()
                print('Target 1 template migrated from template.json to templates.json.')
        self.fig, self.ax = plt.subplots(figsize=figsize)
        first = self.idxs[min(self.k, len(self.idxs) - 1)]
        self.im = self.ax.imshow(get_frame(video_path, first))
        self.ax.axis('off')
        self.gcp_marks, = self.ax.plot([], [], '+', color='lime', markersize=13, mew=2)
        self.target_marks = [self.ax.plot([], [], '+', color=TARGET_COLORS[t % 4],
                                          markersize=15, mew=2)[0] for t in range(n_targets)]
        self.tmpl_lines, self.tmpl_anchor_marks, self.tmpl_handles = [], [], []
        for t in range(n_targets):
            oc, tc = TMPL_COLORS[t % 4], TARGET_COLORS[t % 4]
            self.tmpl_lines.append(self.ax.plot([], [], '-', color=oc, lw=1.5, alpha=0.9, zorder=4)[0])
            self.tmpl_anchor_marks.append(self.ax.plot([], [], 'x', color=tc, ms=11, mew=2, zorder=5)[0])
            self.tmpl_handles.append(self.ax.plot([], [], 'o', mec=oc, mfc=oc, ms=8, alpha=0.7, zorder=5)[0])
        self.fig.canvas.mpl_connect('button_press_event', self._onclick)
        self.fig.canvas.mpl_connect('motion_notify_event', self._onmotion)
        self.fig.canvas.mpl_connect('button_release_event', self._onrelease)
        self.fig.canvas.mpl_connect('key_press_event', self._onkey)
        self._tmpl_redraw()
        self._show_current(reload_img=False)
        plt.show()

    def _current_target_idx(self):
        if self.k >= len(self.idxs):
            return None
        n = len(self.current) - self.n_refs
        return n if 0 <= n < self.n_targets else None

    def _next_label(self):
        n = len(self.current)
        if n < self.n_refs:
            return f'GCP{n}'
        t = n - self.n_refs
        return f'TARGET {t + 1}' if self.n_targets > 1 else 'TARGET'

    def _update_title(self):
        if self.tmpl_def_mode == 'outline':
            et = f'TARGET {self.tmpl_def_target + 1}' if self.n_targets > 1 else 'TARGET'
            self.ax.set_title(f'TEMPLATE {et} -- OUTLINE: {len(self.tmpl_def_pts)} vertices  |  '
                              f't = go to anchor, z = undo vertex')
        elif self.tmpl_def_mode == 'anchor':
            et = f'TARGET {self.tmpl_def_target + 1}' if self.n_targets > 1 else 'TARGET'
            self.ax.set_title(f'TEMPLATE {et} -- click the BASE POINT (anchor)  |  z = back to outline')
        elif self.k >= len(self.idxs):
            n_ok = sum(1 for v in self.records.values() if v is not None)
            self.ax.set_title(f'COMPLETE -- {n_ok} frames recorded, '
                              f'{len(self.idxs) - n_ok} skipped. Run the next cell.')
        else:
            idx = self.idxs[self.k]
            extra = ''
            t = self._current_target_idx()
            if t is not None and self.templates[t]:
                extra = '  ->  move the template and accept with "a"'
            self.ax.set_title(f'Frame {idx} (t={idx/self.fps:.2f}s) -- {self.k+1}/{len(self.idxs)}  |  '
                              f'click {len(self.current)+1}/{self.n_total}: {self._next_label()}{extra}')

    def _redraw_marks(self):
        gcps = self.current[:self.n_refs]
        self.gcp_marks.set_data([p[0] for p in gcps], [p[1] for p in gcps])
        for t in range(self.n_targets):
            i = self.n_refs + t
            pt = self.current[i:i + 1]
            self.target_marks[t].set_data([p[0] for p in pt], [p[1] for p in pt])

    def _show_current(self, reload_img=True):
        if self.k < len(self.idxs) and reload_img:
            self.im.set_data(get_frame(self.video_path, self.idxs[self.k]))
        self._redraw_marks()
        self._update_title()
        self.fig.canvas.draw_idle()

    # --- templates (one per target) ---
    def _tmpl_save(self):
        if self.save_dir:
            with open(self.tmpl_path, 'w', encoding='utf-8') as f:
                json.dump(self.templates, f, indent=1)

    def _tmpl_redraw(self):
        for t in range(self.n_targets):
            if self.tmpl_def_mode and self.tmpl_def_target == t:
                xs = [p[0] for p in self.tmpl_def_pts]
                ys = [p[1] for p in self.tmpl_def_pts]
                if self.tmpl_def_mode == 'anchor' and xs:
                    xs, ys = xs + xs[:1], ys + ys[:1]
                self.tmpl_lines[t].set_data(xs, ys)
                self.tmpl_anchor_marks[t].set_data([], [])
                self.tmpl_handles[t].set_data([], [])
            elif self.templates[t]:
                tpl = self.templates[t]
                ax0, ay0 = tpl['anchor']
                xs = [ax0 + dx for dx, dy in tpl['offsets']]
                ys = [ay0 + dy for dx, dy in tpl['offsets']]
                self.tmpl_lines[t].set_data(xs + xs[:1], ys + ys[:1])
                self.tmpl_anchor_marks[t].set_data([ax0], [ay0])
                self.tmpl_handles[t].set_data([float(np.mean(xs))], [float(np.mean(ys))])
            else:
                self.tmpl_lines[t].set_data([], [])
                self.tmpl_anchor_marks[t].set_data([], [])
                self.tmpl_handles[t].set_data([], [])
        self.fig.canvas.draw_idle()

    def _tmpl_toggle_def(self):
        if not self.tmpl_def_mode:
            t = self._current_target_idx()
            if t is None:
                return
            self.tmpl_def_target = t
            self.tmpl_def_mode = 'outline'
            self.tmpl_def_pts = []
        elif self.tmpl_def_mode == 'outline':
            if len(self.tmpl_def_pts) >= 3:
                self.tmpl_def_mode = 'anchor'
            else:
                print('Template discarded: the outline needs at least 3 vertices.')
                self.tmpl_def_mode = False
                self.tmpl_def_pts = []
                self.tmpl_def_target = None
        else:
            print('Template definition cancelled.')
            self.tmpl_def_mode = False
            self.tmpl_def_pts = []
            self.tmpl_def_target = None
        self._tmpl_redraw()
        self._update_title()
        self.fig.canvas.draw_idle()

    def _tmpl_move(self, ddx, ddy):
        t = self._current_target_idx()
        if t is not None and self.templates[t]:
            self.templates[t]['anchor'][0] += ddx
            self.templates[t]['anchor'][1] += ddy
            self._tmpl_redraw()

    def _tmpl_accept(self):
        t = self._current_target_idx()
        if t is None or self.tmpl_def_mode or not self.templates[t]:
            return
        self._tmpl_save()
        anchor = self.templates[t]['anchor']
        self.current.append([float(anchor[0]), float(anchor[1])])
        if len(self.current) >= self.n_total:
            self._complete_frame(list(self.current))
        else:
            self._show_current(reload_img=False)

    # --- guide lines ---
    def _add_refline(self):
        x0, x1 = self.ax.get_xlim(); y0, y1 = self.ax.get_ylim()
        xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
        dx = (x1 - x0) * 0.2
        line, = self.ax.plot([xm - dx, xm + dx], [ym, ym], '--', color='cyan', lw=1.5, zorder=4)
        h0, = self.ax.plot([xm - dx], [ym], 's', color='cyan', ms=7, zorder=5)
        h1, = self.ax.plot([xm + dx], [ym], 's', color='cyan', ms=7, zorder=5)
        hm, = self.ax.plot([xm], [ym], 'o', mec='cyan', mfc='none', ms=7, mew=1.5, zorder=5)
        self.ref_lines.append({'line': line, 'h0': h0, 'h1': h1, 'hm': hm})
        self.fig.canvas.draw_idle()

    def _remove_last_refline(self):
        if not self.ref_lines:
            return
        rl = self.ref_lines.pop()
        for a in rl.values():
            a.remove()
        self.fig.canvas.draw_idle()

    def _sync_refline(self, rl):
        (x0,), (y0,) = rl['h0'].get_data(); (x1,), (y1,) = rl['h1'].get_data()
        rl['line'].set_data([x0, x1], [y0, y1])
        rl['hm'].set_data([(x0 + x1) / 2], [(y0 + y1) / 2])

    def _hit_handle(self, event):
        if event.x is None or event.y is None:
            return None
        for t in range(self.n_targets):
            hx = self.tmpl_handles[t].get_data()
            if self.templates[t] and len(hx[0]):
                X, Y = self.ax.transData.transform((hx[0][0], hx[1][0]))
                if (X - event.x) ** 2 + (Y - event.y) ** 2 <= HANDLE_TOL_PX ** 2:
                    return ('tmpl', t)
        for rl in reversed(self.ref_lines):
            for which in ('h0', 'h1', 'hm'):
                (px,), (py,) = rl[which].get_data()
                X, Y = self.ax.transData.transform((px, py))
                if (X - event.x) ** 2 + (Y - event.y) ** 2 <= HANDLE_TOL_PX ** 2:
                    return (rl, which)
        return None

    def _onmotion(self, event):
        if self._drag is None or event.xdata is None or event.ydata is None:
            return
        if self._drag[0] == 'tmpl':
            _, t, start_xy, anchor0 = self._drag
            self.templates[t]['anchor'][0] = anchor0[0] + (event.xdata - start_xy[0])
            self.templates[t]['anchor'][1] = anchor0[1] + (event.ydata - start_xy[1])
            self._tmpl_redraw()
            return
        rl, which, start_xy, init = self._drag
        if which == 'hm':
            ddx = event.xdata - start_xy[0]; ddy = event.ydata - start_xy[1]
            (ix0, iy0), (ix1, iy1) = init
            rl['h0'].set_data([ix0 + ddx], [iy0 + ddy])
            rl['h1'].set_data([ix1 + ddx], [iy1 + ddy])
        else:
            rl[which].set_data([event.xdata], [event.ydata])
        self._sync_refline(rl)
        self.fig.canvas.draw_idle()

    def _onrelease(self, event):
        if self._drag is not None and self._drag[0] == 'tmpl':
            self._tmpl_save()
        self._drag = None

    # --- persistence ---
    def _frame_path(self, idx):
        return os.path.join(self.save_dir, f'frame_{idx:06d}.jpg')

    def _write_json(self):
        with open(self.json_path, 'w', encoding='utf-8') as f:
            json.dump({str(k): v for k, v in sorted(self.records.items())}, f, indent=1)

    def _persist(self, idx, pts):
        if not self.save_dir:
            return
        self._write_json()
        img = cv2.cvtColor(get_frame(self.video_path, idx), cv2.COLOR_RGB2BGR)
        if pts is not None:
            for i, (x, y) in enumerate(pts):
                p = (int(round(x)), int(round(y)))
                if i < self.n_refs:
                    cv2.drawMarker(img, p, (0, 255, 0), cv2.MARKER_CROSS, 24, 2)
                    cv2.putText(img, str(i), (p[0] + 6, p[1] - 6),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)
                else:
                    t = i - self.n_refs
                    cv2.drawMarker(img, p, TARGET_BGR[t % 4], cv2.MARKER_CROSS, 30, 2)
            label = f'frame {idx}  t={idx/self.fps:.2f}s'
        else:
            label = f'frame {idx}  t={idx/self.fps:.2f}s  SKIPPED'
        cv2.putText(img, label, (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(img, label, (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.imwrite(self._frame_path(idx), img, [cv2.IMWRITE_JPEG_QUALITY, 95])

    def _unpersist(self, idx):
        if not self.save_dir:
            return
        self._write_json()
        if os.path.exists(self._frame_path(idx)):
            os.remove(self._frame_path(idx))

    # --- events ---
    def _complete_frame(self, pts):
        idx = self.idxs[self.k]
        self.records[idx] = pts
        self._persist(idx, pts)
        self.k += 1
        self.current = []
        self._show_current()

    def _onclick(self, event):
        if event.xdata is None or event.ydata is None:
            return
        if getattr(self.fig.canvas.toolbar, 'mode', ''):
            return
        if self.tmpl_def_mode == 'outline':
            self.tmpl_def_pts.append([float(event.xdata), float(event.ydata)])
            self._tmpl_redraw(); self._update_title(); self.fig.canvas.draw_idle()
            return
        if self.tmpl_def_mode == 'anchor':
            ax0, ay0 = float(event.xdata), float(event.ydata)
            t = self.tmpl_def_target
            self.templates[t] = {'anchor': [ax0, ay0],
                                 'offsets': [[p[0] - ax0, p[1] - ay0] for p in self.tmpl_def_pts]}
            self.tmpl_def_mode = False; self.tmpl_def_pts = []; self.tmpl_def_target = None
            self._tmpl_save(); self._tmpl_redraw(); self._update_title(); self.fig.canvas.draw_idle()
            return
        hit = self._hit_handle(event)
        if hit is not None:
            if hit[0] == 'tmpl':
                _, t = hit
                self._drag = ('tmpl', t, (event.xdata, event.ydata), list(self.templates[t]['anchor']))
            else:
                rl, which = hit
                (x0,), (y0,) = rl['h0'].get_data(); (x1,), (y1,) = rl['h1'].get_data()
                self._drag = (rl, which, (event.xdata, event.ydata), ((x0, y0), (x1, y1)))
            return
        if self.k >= len(self.idxs):
            return
        self.current.append([float(event.xdata), float(event.ydata)])
        if len(self.current) >= self.n_total:
            self._complete_frame(list(self.current))
        else:
            self._show_current(reload_img=False)

    def _onkey(self, event):
        key = event.key or ''
        base = key.replace('shift+', '')
        if key == 't':
            self._tmpl_toggle_def()
        elif self.tmpl_def_mode:
            if key == 'z':
                if self.tmpl_def_mode == 'anchor':
                    self.tmpl_def_mode = 'outline'
                elif self.tmpl_def_pts:
                    self.tmpl_def_pts.pop()
                self._tmpl_redraw(); self._update_title(); self.fig.canvas.draw_idle()
        elif key == 'a':
            self._tmpl_accept()
        elif base in _NUDGE:
            step = 5.0 if key.startswith('shift+') else 1.0
            self._tmpl_move(_NUDGE[base][0] * step, _NUDGE[base][1] * step)
        elif key == 'n' and self.k < len(self.idxs):
            self.current = []
            self._complete_frame(None)
        elif key == 'z':
            if self.current:
                self.current.pop()
                self._show_current(reload_img=False)
            elif self.k > 0:
                self.k -= 1
                idx = self.idxs[self.k]
                self.records.pop(idx, None)
                self._unpersist(idx)
                self._show_current()
        elif key == 'l':
            self._add_refline()
        elif key == 'd':
            self._remove_last_refline()


In [ ]:
class FrameReviewer:
    """Navigates already-resolved frames and lets you correct points by dragging them
    (GCP and targets).

    Keys:  . = next frame   , = previous frame
           a = replace target 1 with its template's anchor
           b = replace target 2 with its template's anchor (if there are 2+ targets)
           arrows = move the ACTIVE template by 1 px (shift+arrow = 5 px) -- the active
                     template is the last one you dragged or accepted with a/b
           l = add a guide line   d = remove the last guide line
    Drag any point (GCP or a target) to correct it. On release, the frame is saved
    immediately (clicks.json + regenerated JPG) and the RMSE in the title is recomputed.
    """
    def __init__(self, picker, figsize=(9, 7)):
        self.p = picker
        self.j = 0
        self._drag = None
        self._active_tmpl = 0
        self.ref_lines = []
        self.fig, self.ax = plt.subplots(figsize=figsize)
        self.im = self.ax.imshow(get_frame(self.p.video_path, self.p.idxs[0]))
        self.ax.axis('off')
        self.gcp_marks, = self.ax.plot([], [], '+', color='lime', ms=13, mew=2, zorder=5)
        self.target_marks = [self.ax.plot([], [], '+', color=TARGET_COLORS[t % 4],
                                          ms=15, mew=2, zorder=5)[0] for t in range(self.p.n_targets)]
        self.labels = []
        self.tmpl_lines, self.tmpl_anchors, self.tmpl_handles = [], [], []
        for t in range(self.p.n_targets):
            oc, tc = TMPL_COLORS[t % 4], TARGET_COLORS[t % 4]
            self.tmpl_lines.append(self.ax.plot([], [], '-', color=oc, lw=1.2, alpha=0.8, zorder=4)[0])
            self.tmpl_anchors.append(self.ax.plot([], [], 'x', color=tc, ms=10, mew=2, zorder=5)[0])
            self.tmpl_handles.append(self.ax.plot([], [], 'o', color=oc, ms=8, alpha=0.7, zorder=5)[0])
        self.fig.canvas.mpl_connect('button_press_event', self._onpress)
        self.fig.canvas.mpl_connect('motion_notify_event', self._onmotion)
        self.fig.canvas.mpl_connect('button_release_event', self._onrelease)
        self.fig.canvas.mpl_connect('key_press_event', self._onkey)
        self._show()
        plt.show()

    def _idx(self):
        return self.p.idxs[self.j]

    def _rec(self):
        return self.p.records.get(self._idx())

    def _rmse(self, rec):
        try:
            g = np.array(rec[:self.p.n_refs], dtype=np.float64)
            Hh = compute_homography(g, gcp_world_arr)
            proj = cv2.perspectiveTransform(g.reshape(-1, 1, 2).astype(np.float32), Hh).reshape(-1, 2)
            return float(np.sqrt(np.mean(np.sum((proj - gcp_world_arr) ** 2, axis=1))))
        except Exception:
            return float('nan')

    def _tmpl_redraw(self):
        for t in range(self.p.n_targets):
            tpl = self.p.templates[t] if t < len(self.p.templates) else None
            if tpl:
                ax0, ay0 = tpl['anchor']
                xs = [ax0 + dx for dx, dy in tpl['offsets']]
                ys = [ay0 + dy for dx, dy in tpl['offsets']]
                self.tmpl_lines[t].set_data(xs + xs[:1], ys + ys[:1])
                self.tmpl_anchors[t].set_data([ax0], [ay0])
                self.tmpl_handles[t].set_data([float(np.mean(xs))], [float(np.mean(ys))])
            else:
                self.tmpl_lines[t].set_data([], [])
                self.tmpl_anchors[t].set_data([], [])
                self.tmpl_handles[t].set_data([], [])

    def _show(self, reload_img=True):
        idx = self._idx()
        if reload_img:
            self.im.set_data(get_frame(self.p.video_path, idx))
        for lab in self.labels:
            lab.remove()
        self.labels = []
        rec = self._rec()
        nav = f'Frame {idx} ({self.j + 1}/{len(self.p.idxs)})'
        if idx not in self.p.records:
            self.gcp_marks.set_data([], [])
            for tm in self.target_marks:
                tm.set_data([], [])
            self.ax.set_title(f'{nav} -- UNRESOLVED (pending in the picker)  |  , . = navigate')
        elif rec is None:
            self.gcp_marks.set_data([], [])
            for tm in self.target_marks:
                tm.set_data([], [])
            self.ax.set_title(f'{nav} -- SKIPPED (use the picker to click it)  |  , . = navigate')
        else:
            self.gcp_marks.set_data([p[0] for p in rec[:self.p.n_refs]],
                                    [p[1] for p in rec[:self.p.n_refs]])
            for t in range(self.p.n_targets):
                i = self.p.n_refs + t
                self.target_marks[t].set_data([rec[i][0]], [rec[i][1]])
            for i, (x, y) in enumerate(rec[:self.p.n_refs]):
                self.labels.append(self.ax.annotate(str(i), (x, y), color='lime', fontsize=9,
                                                    xytext=(6, 6), textcoords='offset points'))
            keys = 'a = target1 to template' + (', b = target2 to template'
                                                if self.p.n_targets > 1 else '')
            self.ax.set_title(f'{nav} -- GCP RMSE: {self._rmse(rec):.3f} m  |  '
                              f', . = navigate | drag a point to correct | {keys}')
        self._tmpl_redraw()
        self.fig.canvas.draw_idle()

    # --- guide lines ---
    def _add_refline(self):
        x0, x1 = self.ax.get_xlim(); y0, y1 = self.ax.get_ylim()
        xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
        dx = (x1 - x0) * 0.2
        line, = self.ax.plot([xm - dx, xm + dx], [ym, ym], '--', color='cyan', lw=1.5, zorder=4)
        h0, = self.ax.plot([xm - dx], [ym], 's', color='cyan', ms=7, zorder=5)
        h1, = self.ax.plot([xm + dx], [ym], 's', color='cyan', ms=7, zorder=5)
        hm, = self.ax.plot([xm], [ym], 'o', mec='cyan', mfc='none', ms=7, mew=1.5, zorder=5)
        self.ref_lines.append({'line': line, 'h0': h0, 'h1': h1, 'hm': hm})
        self.fig.canvas.draw_idle()

    def _remove_last_refline(self):
        if not self.ref_lines:
            return
        rl = self.ref_lines.pop()
        for a in rl.values():
            a.remove()
        self.fig.canvas.draw_idle()

    def _sync_refline(self, rl):
        (x0,), (y0,) = rl['h0'].get_data(); (x1,), (y1,) = rl['h1'].get_data()
        rl['line'].set_data([x0, x1], [y0, y1])
        rl['hm'].set_data([(x0 + x1) / 2], [(y0 + y1) / 2])

    def _hit(self, event):
        rec = self._rec()
        if rec:
            for i, (x, y) in enumerate(rec):
                X, Y = self.ax.transData.transform((x, y))
                if (X - event.x) ** 2 + (Y - event.y) ** 2 <= HANDLE_TOL_PX ** 2:
                    return ('pt', i)
        for t in range(self.p.n_targets):
            hx = self.tmpl_handles[t].get_data()
            if t < len(self.p.templates) and self.p.templates[t] and len(hx[0]):
                X, Y = self.ax.transData.transform((hx[0][0], hx[1][0]))
                if (X - event.x) ** 2 + (Y - event.y) ** 2 <= HANDLE_TOL_PX ** 2:
                    return ('tmpl', t)
        for rl in reversed(self.ref_lines):
            for which in ('h0', 'h1', 'hm'):
                (px,), (py,) = rl[which].get_data()
                X, Y = self.ax.transData.transform((px, py))
                if (X - event.x) ** 2 + (Y - event.y) ** 2 <= HANDLE_TOL_PX ** 2:
                    return ('line', rl, which)
        return None

    def _onpress(self, event):
        if event.xdata is None or getattr(self.fig.canvas.toolbar, 'mode', ''):
            return
        hit = self._hit(event)
        if hit is None:
            return
        if hit[0] == 'pt':
            self._drag = ('pt', hit[1])
        elif hit[0] == 'tmpl':
            t = hit[1]
            self._active_tmpl = t
            self._drag = ('tmpl', t, (event.xdata, event.ydata), list(self.p.templates[t]['anchor']))
        else:
            _, rl, which = hit
            (x0,), (y0,) = rl['h0'].get_data(); (x1,), (y1,) = rl['h1'].get_data()
            self._drag = ('line', rl, which, (event.xdata, event.ydata), ((x0, y0), (x1, y1)))

    def _onmotion(self, event):
        if self._drag is None or event.xdata is None or event.ydata is None:
            return
        if self._drag[0] == 'pt':
            i = self._drag[1]
            rec = self._rec()
            rec[i] = [float(event.xdata), float(event.ydata)]
            if i < self.p.n_refs:
                self.gcp_marks.set_data([p[0] for p in rec[:self.p.n_refs]],
                                        [p[1] for p in rec[:self.p.n_refs]])
                self.labels[i].xy = (rec[i][0], rec[i][1])
            else:
                t = i - self.p.n_refs
                self.target_marks[t].set_data([rec[i][0]], [rec[i][1]])
        elif self._drag[0] == 'tmpl':
            _, t, start_xy, anchor0 = self._drag
            self.p.templates[t]['anchor'][0] = anchor0[0] + (event.xdata - start_xy[0])
            self.p.templates[t]['anchor'][1] = anchor0[1] + (event.ydata - start_xy[1])
            self._tmpl_redraw()
        else:
            _, rl, which, start_xy, init = self._drag
            if which == 'hm':
                ddx = event.xdata - start_xy[0]; ddy = event.ydata - start_xy[1]
                (ix0, iy0), (ix1, iy1) = init
                rl['h0'].set_data([ix0 + ddx], [iy0 + ddy])
                rl['h1'].set_data([ix1 + ddx], [iy1 + ddy])
            else:
                rl[which].set_data([event.xdata], [event.ydata])
            self._sync_refline(rl)
        self.fig.canvas.draw_idle()

    def _onrelease(self, event):
        if self._drag is not None:
            if self._drag[0] == 'pt':
                idx = self._idx()
                self.p._persist(idx, self.p.records[idx])
                self._show(reload_img=False)
            elif self._drag[0] == 'tmpl':
                self.p._tmpl_save()
        self._drag = None

    def _accept_template(self, t):
        rec = self._rec()
        tpl = self.p.templates[t] if t < len(self.p.templates) else None
        if rec and tpl:
            rec[self.p.n_refs + t] = [float(tpl['anchor'][0]), float(tpl['anchor'][1])]
            self._active_tmpl = t
            self.p._persist(self._idx(), rec)
            self._show(reload_img=False)

    def _onkey(self, event):
        key = event.key or ''
        base = key.replace('shift+', '')
        if key == '.':
            self.j = min(self.j + 1, len(self.p.idxs) - 1); self._show()
        elif key == ',':
            self.j = max(self.j - 1, 0); self._show()
        elif key == 'a':
            self._accept_template(0)
        elif key == 'b' and self.p.n_targets > 1:
            self._accept_template(1)
        elif base in _NUDGE:
            t = self._active_tmpl
            tpl = self.p.templates[t] if t < len(self.p.templates) else None
            if tpl:
                step = 5.0 if key.startswith('shift+') else 1.0
                tpl['anchor'][0] += _NUDGE[base][0] * step
                tpl['anchor'][1] += _NUDGE[base][1] * step
                self.p._tmpl_save()
                self._tmpl_redraw()
                self.fig.canvas.draw_idle()
        elif key == 'l':
            self._add_refline()
        elif key == 'd':
            self._remove_last_refline()


In [ ]:
class SegmentMeasurer:
    """Measures real distances (m) of segments clicked on the reference frame, using an
    ALREADY computed homography (H). Every 2 clicks define a segment; its length is
    computed and drawn instantly. Click as many segments as you want (sidewalk width, tile
    footprint, post spacing, etc.) to confirm that the environment's dimensions come out
    with sensible values after scale anchoring.
    Key `z`: undoes the last point (or the last complete segment, if none is half-done).
    The toolbar's zoom/pan does not register clicks while active.
    """
    def __init__(self, img, H, figsize=(9, 7.5)):
        self.H = H
        self.pending_pt = None
        self.pending_marker = None
        self.segments = []  # each: {'p0','p1','length_m','artists':[...]}
        self.fig, self.ax = plt.subplots(figsize=figsize)
        self.ax.imshow(img)
        self.ax.axis('off')
        self.fig.canvas.mpl_connect('button_press_event', self._onclick)
        self.fig.canvas.mpl_connect('key_press_event', self._onkey)
        self._update_title()
        plt.show()

    def _update_title(self):
        n = len(self.segments)
        extra = '  (missing the 2nd click of the current segment)' if self.pending_pt else ''
        self.ax.set_title(f'{n} segment(s) measured{extra}  |  click = point, z = undo')

    def _onclick(self, event):
        if event.xdata is None or event.ydata is None:
            return
        if getattr(self.fig.canvas.toolbar, 'mode', ''):
            return  # zoom/pan active: this click is for navigation, not data
        if self.pending_pt is None:
            self.pending_pt = (event.xdata, event.ydata)
            self.pending_marker, = self.ax.plot(event.xdata, event.ydata, 'y+', ms=14, mew=2)
        else:
            p0, p1 = self.pending_pt, (event.xdata, event.ydata)
            w0 = np.asarray(img_to_world(self.H, p0), float)
            w1 = np.asarray(img_to_world(self.H, p1), float)
            d = float(np.linalg.norm(w1 - w0))
            m1, = self.ax.plot(p1[0], p1[1], 'y+', ms=14, mew=2)
            ln, = self.ax.plot([p0[0], p1[0]], [p0[1], p1[1]], 'y-', lw=1.5)
            xm, ym = (p0[0] + p1[0]) / 2, (p0[1] + p1[1]) / 2
            txt = self.ax.annotate(f'{d:.3f} m', (xm, ym), color='yellow', fontsize=10,
                                    xytext=(4, 4), textcoords='offset points')
            self.segments.append({'p0': p0, 'p1': p1, 'length_m': d,
                                  'artists': [self.pending_marker, m1, ln, txt]})
            print(f'Segment {len(self.segments)}: {d:.3f} m')
            self.pending_pt = None
            self.pending_marker = None
        self._update_title()
        self.fig.canvas.draw_idle()

    def _onkey(self, event):
        if event.key != 'z':
            return
        if self.pending_pt is not None:
            self.pending_marker.remove()
            self.pending_pt = None
            self.pending_marker = None
        elif self.segments:
            seg = self.segments.pop()
            for a in seg['artists']:
                a.remove()
            print(f'Undid segment {len(self.segments) + 1}.')
        self._update_title()
        self.fig.canvas.draw_idle()


In [ ]:
# --- Derived values (run after loading the parameters) ---
_cap = cv2.VideoCapture(VIDEO_PATH)
TOTAL_FRAMES = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS = _cap.get(cv2.CAP_PROP_FPS) or 30.0
_cap.release()

FRAME_STEP = max(1, round(T_STEP_S * FPS))
SAMPLE_IDXS = list(range(FRAME_START, FRAME_END + 1, FRAME_STEP))
if SAMPLE_IDXS[-1] != FRAME_END:
    SAMPLE_IDXS.append(FRAME_END)
SAVE_DIR = CLICK_PATH
frame_ref = get_frame(VIDEO_PATH, FRAME_REF)

print(f'Video: {TOTAL_FRAMES} frames | {FPS:.2f} fps | {TOTAL_FRAMES/FPS:.1f} s')
print(f'Sampling: every {FRAME_STEP} frames ({FRAME_STEP/FPS:.3f} s) -> {len(SAMPLE_IDXS)} frames '
      f'to digitize ({FRAME_START}..{FRAME_END})')
print('Check that the FPS matches the video\'s real duration (CCTV metadata sometimes lies).')


## 1b. Lens distortion correction -- straight-line calibration (recommended)

The video carries no calibration parameters (metadata: codec/FPS only; identified as a
Hikvision DVR, 1024x576 resolution, likely re-encoded). Because it is a wide-angle camera,
radial distortion curves straight lines and the planar homography cannot model that -> bias
toward the frame edges. This is corrected with **plumb-line calibration**: points are
digitized on real-world **straight** elements (kerbs, fences, edges) and the distortion
coefficients that straighten them are estimated, using the video itself (no camera access
needed).

**Implementation:** the **official OpenCV** functions are used throughout --
`cv2.undistortPoints` (iterative Newton inversion) inside the calibration fit, and
`cv2.initUndistortRectifyMap` to build the correction map -- instead of an approximate
closed-form formula. That approximation was tried and discarded: it is only valid for small
distortion and gives 10-30% error at the edges for distortion magnitudes like this video's.
Using the same OpenCV function for calibration and correction guarantees the two stages are
consistent with each other.

**Before inverting, check whether it is needed:** with a guide line over a straight edge that
crosses the frame, see if it bows. If the deviation is a few pixels, distortion is negligible
and you can skip this section.

**How to digitize:** mark 3-6 straight lines, with 4-8 points each, spread across the frame
and **reaching the edges** (where distortion is largest). Once calibrated, `get_frame`
automatically returns corrected frames for the rest of the notebook. This step is optional: if
you skip it, everything runs on the original video.

**Recommended check (avoids overfitting):** digitize one extra straight line that was NOT used
in the calibration, apply the correction to it and confirm visually that it also comes out
straight -- if a held-out control line straightens correctly, the model generalizes instead of
just memorizing the calibration lines.

**Persistence:** once calibrated, k1/k2 are saved to `<video>_distortion.json`. On future runs
(kernel restarted, or running the notebook another day) the calibration is reloaded
automatically and the digitization/fitting cells skip themselves -- no need to re-mark the
lines. To recalibrate (a different video, or to refine it), set `FORCE_RECALIBRATE = True` in
the state cell and re-run from there.

In [ ]:
# --- Distortion calibration persistence ---
DIST_CALIB_PATH = os.path.splitext(VIDEO_PATH)[0] + '_distortion.json'
FORCE_RECALIBRATE = False  # <-- set True to re-digitize/re-fit even if a saved file exists

_dist_cached = None
if os.path.exists(DIST_CALIB_PATH) and not FORCE_RECALIBRATE:
    with open(DIST_CALIB_PATH, encoding='utf-8') as f:
        _dist_cached = json.load(f)
    print(f'Distortion calibration found at: {DIST_CALIB_PATH}')
    print(f"  k1 = {_dist_cached['k1']:+.4f}   k2 = {_dist_cached['k2']:+.4f}   "
          f"({_dist_cached['n_lines']} lines, residual {_dist_cached['residual_before']:.4f} -> "
          f"{_dist_cached['residual']:.6f})")
    print('The next 2 cells skip digitization and fitting automatically.')
    print('To recalibrate (e.g. a different video, or to refine it): set FORCE_RECALIBRATE = True')
    print('and re-run this cell and the next 2.')
else:
    print('No saved calibration for this video (or a forced recompute was requested).')
    print('Continue with the line picker in the next cell.')


In [ ]:
# Digitize on the UNCORRECTED reference frame (always the raw one, even if you re-run this).
_cap = cv2.VideoCapture(VIDEO_PATH); _cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_REF)
_ok, _r = _cap.read(); _cap.release()
_raw_ref = cv2.cvtColor(_r, cv2.COLOR_BGR2RGB)

if _dist_cached is None:
    picker_lines = StraightLinePicker(_raw_ref)
    # <-- mark several straight lines (click=point, n=new line, z=undo). Then run the next cell.
else:
    print('Calibration already loaded from disk: this cell is not needed, you can skip it.')


In [ ]:
# Estimate k1,k2 (standard OpenCV model) that straighten the lines, using cv2.undistortPoints
# -OpenCV's OFFICIAL inversion function (iterative Newton)- inside the fitting objective.
# (An approximate closed-form formula proved inaccurate for real distortion: up to 30% error at
# the edges; cv2.undistortPoints is the reliable reference and is the SAME function used later
# to build the map, so calibration and correction are guaranteed to be consistent.)
H_img, W_img = _raw_ref.shape[:2]
F0 = float(W_img)             # nominal focal length (px); arbitrary, does not affect the later homography
cx, cy = W_img / 2, H_img / 2
K = np.array([[F0, 0, cx], [0, F0, cy], [0, 0, 1]], dtype=np.float64)

if _dist_cached is not None:
    k1, k2 = _dist_cached['k1'], _dist_cached['k2']
    r0, r1 = _dist_cached['residual_before'], _dist_cached['residual']
    dist_coeffs = np.array([k1, k2, 0.0, 0.0], dtype=np.float64)
    print(f'Using saved calibration: k1 = {k1:+.4f}, k2 = {k2:+.4f}  '
          f'(residual {r0:.4f} -> {r1:.6f})')
else:
    from scipy.optimize import least_squares

    lines_px = picker_lines.get_lines()
    assert len(lines_px) >= 2, 'Need >=2 lines with >=3 points each. Add more (key n between lines).'

    def _residuals(k):
        D = np.array([k[0], k[1], 0.0, 0.0], dtype=np.float64)
        out = []
        for ln in lines_px:
            pts = np.asarray(ln, dtype=np.float32).reshape(-1, 1, 2)
            u = cv2.undistortPoints(pts, K, D).reshape(-1, 2)   # -> normalized, ideal coords
            q = u - u.mean(axis=0)
            Vt = np.linalg.svd(q, full_matrices=False)[2]
            along, perp = q @ Vt[0], q @ Vt[1]
            out += list(perp / (np.sqrt(np.mean(along**2)) + 1e-9))   # normalized perpendicular
        return out

    r0 = np.sqrt(np.mean(np.array(_residuals([0.0, 0.0]))**2))
    sol = least_squares(_residuals, [0.0, 0.0])
    k1, k2 = sol.x
    r1 = np.sqrt(np.mean(np.array(_residuals([k1, k2]))**2))
    dist_coeffs = np.array([k1, k2, 0.0, 0.0], dtype=np.float64)

    print(f'Distortion coefficients (OpenCV convention): k1 = {k1:+.4f}, k2 = {k2:+.4f}')
    print(f'Straightness (normalized residual, dimensionless):  before {r0:.4f}  ->  after {r1:.6f}')
    improvement = (r0 - r1) / r0 * 100 if r0 > 0 else 0
    print(f'Residual reduction: {improvement:.1f}%')
    if r1 > 0.02:
        print('(!) Residual still relatively high: check that the clicked lines are really')
        print('    straight in the world, or try adding more/better-distributed points/lines.')
    elif abs(k1) < 0.01:
        print('(!) k1 very small: distortion may be negligible for this video.')

    with open(DIST_CALIB_PATH, 'w', encoding='utf-8') as f:
        json.dump({'k1': float(k1), 'k2': float(k2), 'K': K.tolist(),
                   'W_img': int(W_img), 'H_img': int(H_img), 'n_lines': len(lines_px),
                   'residual_before': float(r0), 'residual': float(r1)}, f, indent=1)
    print(f'\nCalibration saved to: {DIST_CALIB_PATH}')
    print('It will be reused automatically on future runs (without re-digitizing).')


In [ ]:
# Build the correction map with OpenCV's OFFICIAL function (the same one validated above)
# and ENABLE undistortion in get_frame for the rest of the notebook.
map1, map2 = cv2.initUndistortRectifyMap(K, dist_coeffs, None, K, (W_img, H_img), cv2.CV_32FC1)
_UNDIST_MAPS = (map1, map2)
frame_ref = get_frame(VIDEO_PATH, FRAME_REF)   # reload already corrected, for section 3 (GCP)

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].imshow(_raw_ref); axs[0].set_title('Original (distorted)'); axs[0].axis('off')
if _dist_cached is None:
    for ln in lines_px:
        a = np.array(ln); axs[0].plot(a[:, 0], a[:, 1], '-o', ms=3, lw=1)
axs[1].imshow(frame_ref); axs[1].set_title('Corrected (undistorted)'); axs[1].axis('off')
plt.tight_layout()
save_fig(fig, 'fig02_distortion_correction',
         'Comparison of a reference frame before and after lens distortion correction.')
print('Correction ACTIVE: get_frame now returns corrected frames. Continue with section 3 (GCP).')
print('Note: if you re-run the base-functions cell, re-run this section too to reactivate it.')
print('Recommended check: click a straight line NOT used in the calibration and confirm')
print('visually that it also came out straight (avoids overfitting to the calibration lines).')


## 2. Frame explorer (optional)

Move the slider to locate frames of interest and note their indices in `FRAME_START` /
`FRAME_END` / `FRAME_REF` (parameters cell). Displays JPEG in a widget (no matplotlib) --
smooth and immune to backend conflicts.

In [ ]:
# Explorer WITHOUT matplotlib: shows the frame in a widgets.Image (JPEG).
# Immune to Qt-backend issues and much smoother when moving the slider.
_cap = cv2.VideoCapture(VIDEO_PATH)
TOTAL_FRAMES = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS = _cap.get(cv2.CAP_PROP_FPS) or 30.0
print(f'# explorer -- Total frames: {TOTAL_FRAMES}  |  FPS: {FPS:.2f}  |  Approx. duration: {TOTAL_FRAMES/FPS:.1f} s')

def _fmt_time(sec):
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = sec % 60
    return f'{h:02d}:{m:02d}:{s:05.2f}'

_label = widgets.Label()
_img_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='820px'))

def _show_frame(idx):
    _cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = _cap.read()  # BGR, which is exactly what imencode expects
    if not ret:
        _label.value = f'Could not read frame {idx}'
        return
    ok, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
    if ok:
        _img_widget.value = buf.tobytes()
    t = idx / FPS
    _label.value = f'Frame {idx} / {TOTAL_FRAMES-1}   |   t = {t:.2f}s ({_fmt_time(t)})'

_slider = widgets.IntSlider(min=0, max=max(TOTAL_FRAMES-1, 0), step=1, value=0,
                             description='Frame:', continuous_update=False,
                             layout=widgets.Layout(width='600px'))
_slider.observe(lambda change: _show_frame(change['new']), names='value')

display(_slider, _label, _img_widget)
_show_frame(_slider.value)


## 3. Ground control points (GCP) on the reference frame

Click `N_GCP` identifiable, **coplanar** points (at ground level: TDP bases, tile or
fence corners), well spread out and non-collinear. **Remember the order** of your clicks: you
will need it in step 4.

**Fixing a bad click:** key `z` undoes the last clicked point (works even after all N clicks
are done). **Guide lines (optional):** key `l` adds a draggable dotted line (endpoints = orient,
centre circle = translate) to help you place points precisely (e.g. align with a straight
edge); `d` removes the last one. They are a visual aid only, not recorded as data. The
toolbar's zoom/pan does not register clicks while active.

In [ ]:
# Click the N_GCP control points IN ORDER on the reference frame.
# Choose them coplanar (at ground level), well spread out and non-collinear.
picker_gcp = PointPicker(frame_ref, N_GCP, 'GCP on the reference frame (IN ORDER, with zoom)')
# <-- click, then run the next cell.


In [ ]:
gcp_img_ref = picker_gcp.points
assert len(gcp_img_ref) == N_GCP, f'There are {len(gcp_img_ref)} clicks; {N_GCP} are needed.'
print('GCP in image coordinates (reference frame):')
for i, p in enumerate(gcp_img_ref):
    print(f'  GCP{i}: ({p[0]:.1f}, {p[1]:.1f}) px')


## 4. Real-world coordinates of the GCP (.gpkg -> local UTM)

The GCP are read in lat/long, projected to UTM and expressed in local metres. Then, in
`REF_POINT_ORDER`, map each click (in your own order) to the matching point index in the file.
Resulting axes: **X = East, Y = North** (UTM grid north).

In [ ]:
import geopandas as gpd

def azimuth_from_north(direction_xy):
    """direction_xy: vector (dE, dN). Azimuth measured from North (+Y), clockwise."""
    dE, dN = direction_xy
    return np.arctan2(dE, dN)

def fit_line_direction(points):
    """Line fit by PCA (least squares) over a set of collinear points.
    Returns the unit vector of the principal direction."""
    centered = points - points.mean(axis=0)
    _, _, vt = np.linalg.svd(centered)
    return vt[0]

# --- 1) Existing GCP (.gpkg): true reference azimuth from points 1, 4, 5 ---
gdf = gpd.read_file(REF_POINTS_PATH_QGIS)
lon_m, lat_m = gdf.geometry.x.mean(), gdf.geometry.y.mean()
utm_zone = int((lon_m + 180) / 6) + 1
epsg_utm = (32600 if lat_m >= 0 else 32700) + utm_zone

xy_utm = np.array([(g.x, g.y) for g in gdf.to_crs(epsg=epsg_utm).geometry])
xy_local = xy_utm - xy_utm[0]

IDX_AZ = [0, 3, 4]   # gpkg points that define the true azimuth (collinear)
dir_target = fit_line_direction(xy_local[IDX_AZ])
az_target = azimuth_from_north(dir_target)
print(f'True reference azimuth (gpkg, pts {IDX_AZ}, from North): {np.degrees(az_target)%360:.3f} deg')

# --- 2) New points (in situ, already extracted from the DXF to txt): id, x, y in their own frame ---
data = np.loadtxt(REF_POINTS_PATH_INSITU, skiprows=1)
xy_insitu = data[:, 1:3]
print(f'{len(xy_insitu)} points loaded from {REF_POINTS_PATH_INSITU}')

# --- 3) Direction of the Y=0 points (baseline to rotate) ---
BASELINE_MASK = np.isclose(xy_insitu[:, 1], 0, atol=1e-3)
assert BASELINE_MASK.sum() >= 2, 'Adjust BASELINE_MASK: not enough points found at Y=0.'
dir_insitu = fit_line_direction(xy_insitu[BASELINE_MASK])
az_insitu = azimuth_from_north(dir_insitu)
print(f'In-situ baseline points used (Y=0): {np.where(BASELINE_MASK)[0].tolist()}')
print(f'In-situ baseline azimuth (own frame): {np.degrees(az_insitu)%360:.3f} deg')

# --- 4) Rotation + translation (in-situ origin (0,0) == gpkg point 0 == (0,0) in xy_local) ---
theta = -(az_target - az_insitu)
print(f'Applied rotation: {np.degrees(theta):.3f} deg')

R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
xy_insitu_final = xy_insitu @ R.T + xy_local[0]

# --- 5) Save ONLY the already-rotated result ---
out_path = os.path.join(BASE_PATH, "ref_points_dxf_rotated.txt")
with open(out_path, 'w') as f:
    f.write('id\tE_m\tN_m\n')
    for i, (x, y) in enumerate(xy_insitu_final):
        f.write(f'{i}\t{x:.4f}\t{y:.4f}\n')
print(f'\nSaved: {out_path}')
for i, p in enumerate(xy_insitu_final):
    print(f'  point d{i}: E={p[0]:8.3f}  N={p[1]:8.3f}')

# --- 6) Visual check (optional) ---
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(xy_local[:, 0], xy_local[:, 1], 'o-', color='blue', label='GCP (gpkg)', ms=6)
ax.plot(xy_insitu_final[:, 0], xy_insitu_final[:, 1], 'x--', color='orange', label='Rotated in-situ points', ms=6)
for i, (x, y) in enumerate(xy_local):
    ax.annotate(f'GCP{i}', (x, y), textcoords="offset points", xytext=(5,5), ha='left', fontsize=9, color='blue')
for i, (x, y) in enumerate(xy_insitu_final):
    ax.annotate(f'd{i}', (x, y), textcoords="offset points", xytext=(5,5), ha='left', fontsize=9, color='orange')
ax.set_aspect('equal')
ax.set_xlabel('East (m)'); ax.set_ylabel('North (m)')
ax.legend()
save_fig(fig, 'fig03a_gcp_dxf_rotation',
         'Satellite GCP (gpkg) vs. in-situ points (DXF) after rotation/translation to the unified frame.')


In [ ]:
# --- In-situ points only (already rotated) as the GCP source ---
xy_all = xy_insitu_final
n_insitu = len(xy_insitu_final)

print(f'Points available as GCP: {n_insitu} (in situ)')
for i, p in enumerate(xy_all):
    print(f'  point {i} (in situ): E={p[0]:8.3f}  N={p[1]:8.3f}')

# For each GCP click, IN THE ORDER you clicked them (section 3), give the index of the
# matching point (within xy_all) from the file.
if VIEW_NAME == 'Vista_1':
    REF_POINT_ORDER = [0, 1, 2, 3, 4, 5]   # <-- ADJUST to your case
elif VIEW_NAME == 'Vista_2':
    REF_POINT_ORDER = [0, 1, 2, 3, 4, 5, 6]   # <-- ADJUST to your case

assert len(REF_POINT_ORDER) == N_GCP, f'REF_POINT_ORDER must have {N_GCP} indices.'
assert max(REF_POINT_ORDER) < len(xy_all), 'Index out of range in REF_POINT_ORDER.'
gcp_world_arr = np.array([xy_all[i] for i in REF_POINT_ORDER], dtype=np.float32)
print('\ngcp_world (m), in the same order as the clicks:')
for i, p in enumerate(gcp_world_arr):
    print(f'  GCP{i}: E={p[0]:.3f}  N={p[1]:.3f}')

# --- 7) Visual check: plot of the GCP (clicks) against their real-world coordinates (m) ---
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(gcp_world_arr[:, 0], gcp_world_arr[:, 1], 'o-', color='green', label='GCP (m)', ms=6)
for i, (x, y) in enumerate(gcp_world_arr):
    ax.annotate(f'GCP{i}', (x, y), textcoords="offset points", xytext=(5,5), ha='left', fontsize=9, color='green')
ax.set_aspect('equal')
ax.set_xlabel('East (m)'); ax.set_ylabel('North (m)')
ax.set_title('GCP in real-world coordinates (m) -- same order as the clicks')
ax.legend()
save_fig(fig, 'fig03_gcp_world',
         'Ground control points (GCP) in the local world frame (m), in the same order as the clicks.')


## 5. Sanity check: measuring ground lengths

With the scale already anchored, freely measure distances on ground elements (sidewalk width,
tile footprint, post spacing, lane width, etc.) and confirm they come out with sensible values
-- a qualitative check, complementary to the numeric one in section 6.

**Usage:** every 2 clicks define a segment; its length in metres is computed and printed
instantly, and drawn on the image. You can measure as many segments as you like. Key `z` undoes
the last point (or the last complete segment if none is half-done). If a measurement doesn't
make sense (e.g. a "40 cm" or "8 m" sidewalk), check `REF_POINT_ORDER`, GCP coplanarity, or the
scale anchoring before continuing.

In [ ]:
H_check = compute_homography(gcp_img_ref, gcp_world_arr)
measurer = SegmentMeasurer(frame_ref, H_check)
# <-- click pairs of points on ground elements of recognizable size.
#     Every 2 clicks prints the measured distance. z = undo. No limit on segments.


## 6. Homography verification (automatic)

Two click-free checks: **GCP reprojection RMSE** and **all pairwise distances** (real vs.
measured). Discrepancies > 10% usually point to a wrong `REF_POINT_ORDER` or non-coplanar GCP.
Fix it before continuing.

In [ ]:
# Automatic check (no clicks): GCP reprojection and pairwise distances.
H_ref = compute_homography(gcp_img_ref, gcp_world_arr)
proj = cv2.perspectiveTransform(np.array(gcp_img_ref, np.float32).reshape(-1,1,2), H_ref).reshape(-1,2)
err = np.linalg.norm(proj - gcp_world_arr, axis=1)
print(f'GCP reprojection RMSE: {np.sqrt(np.mean(err**2)):.3f} m (max {err.max():.3f} m)')
if N_GCP <= 4:
    print('(with 4 GCP the fit is exact by construction; use 5+ for this check to be informative)')

worst = 1.0
print('\nPairwise distances -- real (file, already anchored) vs. measured (homography):')
for i in range(N_GCP):
    for j in range(i+1, N_GCP):
        dr = np.linalg.norm(gcp_world_arr[i]-gcp_world_arr[j])
        dm = np.linalg.norm(proj[i]-proj[j])
        r = dm/dr if dr > 0 else np.nan
        worst = max(worst, r, 1/r) if dr > 0 else worst
        print(f'  GCP{i}-GCP{j}: real {dr:6.2f} m  measured {dm:6.2f} m  ratio {r:.3f}')
print('\n' + ('*** CHECK: discrepancy > 10% on some pair (REF_POINT_ORDER? coplanarity?) ***'
              if worst > 1.10 else 'OK: all GCP pairwise distances match within 10%.'))


## 7. Per-frame digitization (interactive)

For every sampled frame: the `N_GCP` GCP **in the same order** as in step 3, then
**`N_TARGETS` target(s) to track** (e.g. one tracked TDP). The title always shows which point
is next ("GCP2", "TARGET 1", "TARGET 2", ...).

**Per-target template:** each target has its own template (outline + anchor point), with a
distinct colour so they don't get mixed up (target 1 = red/yellow, target 2 = blue/magenta).
Key `t` draws the outline **of the target currently being clicked**; `t` again switches to the
anchor phase; the next click closes it. Key `a` accepts the active template's anchor and
advances on its own to the next target (or the next frame, if it was the last one). See the
full key list in the cell's comment.

**Compatibility:** if you already had frames digitized with a single target (old format), they
are migrated automatically on load: the new target lands on a fill-in point `(20, 20)` that's
easy to spot -- you'll see a message with how many frames were affected -- go through them in
section 8 with target 2's template.

In [ ]:
class TemplateEditor:
    """Independently creates or edits the template (outline + anchor) of ONE target, on a
    single reference frame. Compatible with the templates.json format used by SequencePicker
    and FrameReviewer -- you can run this before section 7 to have the target's template
    already defined, or to redo it if it ended up poorly fitted.

    Keys:
      click   = adds a vertex to the outline (min. 3)
      t       = closes the outline and switches to ANCHOR mode; the next click sets the base point
      z       = undoes the last vertex (in outline mode) or goes back to the outline (in anchor mode)
      e       = if a template already exists for this target, loads its vertices for editing
                from scratch (starts from a blank template if none exists)
      arrows  = moves the already-set ANCHOR by 1 px (shift+arrow = 5 px)
    Saves immediately to templates.json on acceptance (no need to close the figure).
    """
    def __init__(self, frame, target_idx, save_dir, n_targets, figsize=(9, 7)):
        self.target_idx = target_idx
        self.n_targets = n_targets
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.tmpl_path = os.path.join(save_dir, 'templates.json')

        if os.path.exists(self.tmpl_path):
            with open(self.tmpl_path, encoding='utf-8') as f:
                loaded = json.load(f)
        else:
            loaded = []
        self.templates = (loaded + [None] * n_targets)[:n_targets]

        self.mode = 'outline'   # 'outline' | 'anchor'
        self.pts = []

        oc = TMPL_COLORS[target_idx % 4]
        tc = TARGET_COLORS[target_idx % 4]

        self.fig, self.ax = plt.subplots(figsize=figsize)
        self.ax.imshow(frame)
        self.ax.axis('off')
        self.line, = self.ax.plot([], [], '-', color=oc, lw=1.5, zorder=4)
        self.anchor_mark, = self.ax.plot([], [], 'x', color=tc, ms=13, mew=2, zorder=5)

        self.fig.canvas.mpl_connect('button_press_event', self._onclick)
        self.fig.canvas.mpl_connect('key_press_event', self._onkey)
        self._update_title()
        plt.show()

    def _update_title(self):
        if self.mode == 'outline':
            self.ax.set_title(f'Target {self.target_idx + 1} -- OUTLINE: {len(self.pts)} vertices  |  '
                              f't = go to anchor (min. 3)  |  e = load existing template')
        else:
            self.ax.set_title(f'Target {self.target_idx + 1} -- click the BASE POINT (anchor)  |  '
                              f'z = back to outline')
        self.fig.canvas.draw_idle()

    def _redraw(self):
        xs = [p[0] for p in self.pts]
        ys = [p[1] for p in self.pts]
        if self.mode == 'anchor' and xs:
            xs, ys = xs + xs[:1], ys + ys[:1]
        self.line.set_data(xs, ys)
        self.fig.canvas.draw_idle()

    def _onclick(self, event):
        if event.xdata is None or event.ydata is None or event.inaxes != self.ax:
            return
        if self.mode == 'outline':
            self.pts.append([event.xdata, event.ydata])
            self._redraw()
            self._update_title()
        elif self.mode == 'anchor':
            anchor = [float(event.xdata), float(event.ydata)]
            offsets = [[p[0] - anchor[0], p[1] - anchor[1]] for p in self.pts]
            self.templates[self.target_idx] = {'anchor': anchor, 'offsets': offsets}
            self._save()
            self.anchor_mark.set_data([anchor[0]], [anchor[1]])
            self.ax.set_title(f'Target {self.target_idx + 1} -- template saved to templates.json '
                              f'({len(self.pts)} vertices). Close the figure or define another target.')
            self.fig.canvas.draw_idle()

    def _onkey(self, event):
        if event.key == 't' and self.mode == 'outline':
            if len(self.pts) >= 3:
                self.mode = 'anchor'
                self._update_title()
            else:
                print('You need at least 3 vertices before moving to the anchor.')
        elif event.key == 'z':
            if self.mode == 'anchor':
                self.mode = 'outline'
                self._update_title()
            elif self.pts:
                self.pts.pop()
                self._redraw()
                self._update_title()
        elif event.key == 'e' and self.mode == 'outline' and not self.pts:
            tpl = self.templates[self.target_idx]
            if tpl:
                ax0, ay0 = tpl['anchor']
                self.pts = [[ax0 + dx, ay0 + dy] for dx, dy in tpl['offsets']]
                self._redraw()
                self._update_title()
                print(f'Existing template loaded ({len(self.pts)} vertices) -- edit it and accept again.')
            else:
                print('No template saved yet for this target.')
        elif event.key in _NUDGE and self.templates[self.target_idx]:
            ddx, ddy = _NUDGE[event.key]
            # note: matplotlib does not report the shift modifier in event.key for arrow keys by
            # default; use plain arrows (1 px) -- fine-tune with more anchor clicks if you need more.
            self.templates[self.target_idx]['anchor'][0] += ddx
            self.templates[self.target_idx]['anchor'][1] += ddy
            self._save()
            ax0, ay0 = self.templates[self.target_idx]['anchor']
            self.anchor_mark.set_data([ax0], [ay0])
            self.fig.canvas.draw_idle()

    def _save(self):
        with open(self.tmpl_path, 'w', encoding='utf-8') as f:
            json.dump(self.templates, f, indent=1)


# Draws the target's outline on the reference frame: click per vertex, 't' to move to the
# anchor, next click sets the base point. Saves on its own.
TARGET_IDX_TEMPLATE = 0   # 0 = target 1, 1 = target 2, etc.
tmpl_editor = TemplateEditor(frame_ref, TARGET_IDX_TEMPLATE, SAVE_DIR, N_TARGETS)


In [ ]:
# Digitize every frame: the N_GCP GCP (same order as section 3) + N_TARGETS target(s).
# A homography is computed PER FRAME -> immune to camera motion.
# For each target, use its TEMPLATE: key t draws the outline (of the target currently being
# clicked), t again and the next click sets the base point (anchor); then on each frame drag
# the silhouette into place and accept with a (advances on its own to the next target/frame).
# Keys: n=skip frame | z=undo | l=guide line | d=remove line | arrows=move template
picker_seq = SequencePicker(VIDEO_PATH, SAMPLE_IDXS, FPS, n_refs=N_GCP, n_targets=N_TARGETS, save_dir=SAVE_DIR)
print(f'Saved instantly to {SAVE_DIR} (resumable). Once it says COMPLETE, continue.')


## 8. Review and correction (interactive, optional)

Navigate the frames and correct any point (GCP or target) by dragging it; the per-frame RMSE
(computed from the GCP only) recomputes live. Besides dragging directly, you can use each
target's template: key `a` applies target 1's template, key `b` applies target 2's (drag them
by their handle or nudge with arrow keys -- the "active" one for the arrow keys is whichever
you last touched). This is the recommended place to reposition targets migrated from the old
format (section 7).

In [ ]:
# Review frame by frame; drag any point to correct it (saved on release) and the title's
# RMSE recomputes. Keys: , . = navigate | a = target to anchor | l/d = guide lines.
reviewer = FrameReviewer(picker_seq)


## 9. Trajectory (per-frame homography)

Converts every frame to world coordinates with its own homography, builds the time series
(E/N position, cumulative displacement), prints the per-frame RMSE as a quality check, and
saves a CSV.

In [ ]:
# --- Time series: PER-FRAME homography, conversion to world coordinates, table and plots ---
missing = [idx for idx in SAMPLE_IDXS if idx not in picker_seq.records]
assert not missing, f'{len(missing)} frames still need clicking/skipping (starting at frame {missing[0]}).'

valid = [(idx, rec) for idx in SAMPLE_IDXS if (rec := picker_seq.records.get(idx)) is not None]
assert len(valid) >= 2, 'At least 2 recorded frames are needed.'

t_serie = np.array([idx / FPS for idx, _ in valid])
gcp_img_serie = np.array([rec[:N_GCP] for _, rec in valid], dtype=np.float64)   # (n, N_GCP, 2)
tdp_img_serie = np.array([rec[N_GCP] for _, rec in valid], dtype=np.float64)   # (n, 2)

# One homography per frame (corrects camera motion) + reprojection error as QC
pts_world_serie = np.empty((len(valid), 2))
rmse_serie = np.empty(len(valid))
H_serie = []
for i, (idx, _) in enumerate(valid):
    H_i = compute_homography(gcp_img_serie[i], gcp_world_arr)
    H_serie.append(H_i)
    pts_world_serie[i] = img_to_world(H_i, tdp_img_serie[i])
    proj = cv2.perspectiveTransform(
        gcp_img_serie[i].reshape(-1, 1, 2).astype(np.float32), H_i).reshape(-1, 2)
    rmse_serie[i] = np.sqrt(np.mean(np.sum((proj - gcp_world_arr) ** 2, axis=1)))

# Position relative to the first sample and cumulative displacement
rel_serie = pts_world_serie - pts_world_serie[0]
disp_serie = np.linalg.norm(rel_serie, axis=1)

RMSE_SUSPICIOUS = 2 * np.median(rmse_serie)
print(f'{len(valid)} valid samples out of {len(SAMPLE_IDXS)}:')
print(f'{"frame":>6} {"t (s)":>8} {"E (m)":>9} {"N (m)":>9} {"|d| cum. (m)":>13} {"GCP RMSE (m)":>13}')
for (idx, _), tt, (x, y), d, e in zip(valid, t_serie, pts_world_serie, disp_serie, rmse_serie):
    flag = '  <-- check' if e > RMSE_SUSPICIOUS else ''
    print(f'{idx:>6} {tt:>8.2f} {x:>9.3f} {y:>9.3f} {d:>13.3f} {e:>13.3f}{flag}')
print(f'\nTotal displacement (first -> last sample): {disp_serie[-1]:.3f} m')
print('The per-frame RMSE measures how well your GCP clicks match the real-world coordinates;')
print('a frame with RMSE well above the median indicates bad clicks or a misidentified GCP (re-click it).')

# Save to CSV next to the video, for later analysis
import csv
csv_path = os.path.splitext(VIDEO_PATH)[0] + '_tdp_trajectory.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['frame', 't_s', 'px_x', 'px_y', 'east_m', 'north_m', 'cum_disp_m', 'gcp_rmse_m'])
    for (idx, _), tt, pxy, (x, y), d, e in zip(valid, t_serie, tdp_img_serie,
                                               pts_world_serie, disp_serie, rmse_serie):
        w.writerow([idx, f'{tt:.3f}', f'{pxy[0]:.1f}', f'{pxy[1]:.1f}',
                    f'{x:.4f}', f'{y:.4f}', f'{d:.4f}', f'{e:.4f}'])
print(f'Series saved to: {csv_path}')

fig, axs = plt.subplots(1, 3, figsize=(15.5, 4.6))

# (1) Plan-view trajectory, coloured by time
sc = axs[0].scatter(pts_world_serie[:, 0], pts_world_serie[:, 1], c=t_serie, cmap='viridis', s=30, zorder=3)
axs[0].plot(pts_world_serie[:, 0], pts_world_serie[:, 1], '-', color='gray', lw=0.8, alpha=0.6)
axs[0].scatter(gcp_world_arr[:, 0], gcp_world_arr[:, 1], c='green', marker='s', s=40, label='GCP')
fig.colorbar(sc, ax=axs[0], label='t (s)')
axs[0].set_title('Plan-view trajectory')
axs[0].set_xlabel('East (m)'); axs[0].set_ylabel('North (m)')
axs[0].axis('equal'); axs[0].legend(); axs[0].grid(alpha=0.3)

# (2) Relative position components vs. time
axs[1].plot(t_serie, rel_serie[:, 0], 'o-', ms=4, label='dE -- East (m)')
axs[1].plot(t_serie, rel_serie[:, 1], 'o-', ms=4, label='dN -- North (m)')
axs[1].set_title('Position relative to the start')
axs[1].set_xlabel('t (s)'); axs[1].set_ylabel('displacement (m)')
axs[1].legend(); axs[1].grid(alpha=0.3)

# (3) Cumulative displacement magnitude vs. time
axs[2].plot(t_serie, disp_serie, 'o-', ms=4, color='crimson')
axs[2].set_title('Cumulative |d| relative to the start')
axs[2].set_xlabel('t (s)'); axs[2].set_ylabel('|d| (m)')
axs[2].grid(alpha=0.3)

plt.tight_layout()
save_fig(fig, 'fig05_raw_trajectory',
         'Plan-view trajectory, relative position components and cumulative displacement (unfiltered trace).')

# save the raw trace, in 3 columns: time, E and N (m) -- filename matches what
# fusion_BLUE.ipynb expects from raw_trace_path().
raw_path = os.path.join(OUTPUT_PATH, f'{VIEW_NAME}_{TARGET}_raw_trace.txt')
np.savetxt(raw_path, np.column_stack((t_serie, pts_world_serie)), fmt='%.4f', header='t_s east_m north_m', comments='')
print(f'Raw trace saved to: {raw_path}')


## 10. Uncertainty (Monte Carlo)

Perturbs the clicks (GCP and TDP, per frame) and the GCP coordinates, and recomputes the
trajectory `N_SIM` times. Produces the plan-view trace with 2-sigma ellipses and the
components with 50% and 95% bands. The solid line is the **nominal** value (unperturbed
clicks).

This is the notebook's final product: `mc_result_{VIEW_NAME}_{TARGET}.npz` (filtered position
trace + per-frame covariance), consumed downstream by `fusion_BLUE.ipynb`.

In [ ]:
from matplotlib.patches import Ellipse
from scipy.signal import butter, filtfilt

N_SIM_SERIE = N_SIM
if 'SIGMA_CLICK_PX' not in globals():
    SIGMA_CLICK_PX = 2.0
if 'SIGMA_GCP_WORLD_M' not in globals():
    SIGMA_GCP_WORLD_M = 0.3
rng = np.random.default_rng(7)

_w = np.asarray(gcp_world_arr, dtype=np.float64)
n_pts = len(tdp_img_serie)

sims_world = np.empty((N_SIM_SERIE, n_pts, 2))
sims_rel = np.empty_like(sims_world)
for k in range(N_SIM_SERIE):
    w_k = _w + rng.normal(0, SIGMA_GCP_WORLD_M, _w.shape) if SIGMA_GCP_WORLD_M > 0 else _w
    for i in range(n_pts):
        g_i = gcp_img_serie[i] + rng.normal(0, SIGMA_CLICK_PX, gcp_img_serie[i].shape)
        H_ik = compute_homography(g_i, w_k)
        c_i = tdp_img_serie[i] + rng.normal(0, SIGMA_CLICK_PX, 2)
        sims_world[k, i] = img_to_world(H_ik, c_i)
    sims_rel[k] = sims_world[k] - sims_world[k, 0]

# --- keep the raw nominal trace BEFORE filtering ---
rel_serie_raw = rel_serie.copy()
disp_serie_raw = np.linalg.norm(rel_serie_raw, axis=1)

# --- Low-pass (Butterworth, zero-phase) -- free cutoff parameter ---
CUTOFF_HZ = 2   # <-- adjust (Hz); None = no filtering
FILT_ORDER = 4

def lowpass(x, t, cutoff_hz, order=FILT_ORDER, axis=0):
    if cutoff_hz is None:
        return x
    dt = np.median(np.diff(t))
    fs = 1.0 / dt
    nyq = 0.5 * fs
    if cutoff_hz >= nyq:
        raise ValueError(f'CUTOFF_HZ={cutoff_hz} >= Nyquist={nyq:.3f} Hz (fs~{fs:.2f} Hz). Lower the cutoff.')
    b, a = butter(order, cutoff_hz / nyq, btype='low')
    return filtfilt(b, a, x, axis=axis)

sims_world = lowpass(sims_world, t_serie, CUTOFF_HZ, axis=1)
sims_rel   = lowpass(sims_rel,   t_serie, CUTOFF_HZ, axis=1)
pts_world_serie = lowpass(pts_world_serie, t_serie, CUTOFF_HZ, axis=0)
rel_serie = rel_serie_raw if CUTOFF_HZ is None else lowpass(rel_serie_raw, t_serie, CUTOFF_HZ, axis=0)

sims_disp = np.linalg.norm(sims_rel, axis=2)
disp_serie = np.linalg.norm(rel_serie, axis=1)

rel_lo50, rel_hi50 = np.percentile(sims_rel, [25, 75], axis=0)
rel_lo95, rel_hi95 = np.percentile(sims_rel, [2.5, 97.5], axis=0)
disp_lo50, disp_hi50 = np.percentile(sims_disp, [25, 75], axis=0)
disp_lo95, disp_hi95 = np.percentile(sims_disp, [2.5, 97.5], axis=0)
sigma_per_point = sims_world.std(axis=0).mean(axis=1)

print(f'Trace Monte Carlo: {N_SIM_SERIE} simulations, per-frame homography, '
      f'sigma_click = {SIGMA_CLICK_PX} px, sigma_ref = {SIGMA_GCP_WORLD_M} m, '
      f'lowpass = {CUTOFF_HZ} Hz')
print(f'Per-point position uncertainty (mean sigma): '
      f'{sigma_per_point.min():.3f} - {sigma_per_point.max():.3f} m')
print(f'Total displacement: {disp_serie[-1]:.3f} m   '
      f'(50%: [{disp_lo50[-1]:.3f}, {disp_hi50[-1]:.3f}] m | '
      f'95%: [{disp_lo95[-1]:.3f}, {disp_hi95[-1]:.3f}] m)')

fig, axs = plt.subplots(1, 2, figsize=(12.5, 5))

sc = axs[0].scatter(pts_world_serie[:, 0] - pts_world_serie[0, 0], pts_world_serie[:, 1] - pts_world_serie[0, 1], c=t_serie,
                    cmap='viridis', s=30, zorder=3)
axs[0].plot(pts_world_serie[:, 0] - pts_world_serie[0, 0], pts_world_serie[:, 1] - pts_world_serie[0, 1], '-', color='gray', lw=0.8, alpha=0.6)
fig.colorbar(sc, ax=axs[0], label='t (s)')
axs[0].set_title('Plan-view trace (filtered)')
axs[0].set_xlabel('East (m)'); axs[0].set_ylabel('North (m)')
axs[0].axis('equal'); axs[0].grid(alpha=0.3)

axs[1].fill_between(t_serie, rel_lo95[:, 0], rel_hi95[:, 0], alpha=0.12, color='C0', label='dE 95% band')
axs[1].fill_between(t_serie, rel_lo50[:, 0], rel_hi50[:, 0], alpha=0.35, color='C0', label='dE 50% band')
axs[1].plot(t_serie, rel_serie[:, 0], 'o-', ms=4, color='C0', label='dE nominal (East)')
axs[1].fill_between(t_serie, rel_lo95[:, 1], rel_hi95[:, 1], alpha=0.12, color='C1', label='dN 95% band')
axs[1].fill_between(t_serie, rel_lo50[:, 1], rel_hi50[:, 1], alpha=0.35, color='C1', label='dN 50% band')
axs[1].plot(t_serie, rel_serie[:, 1], 'o-', ms=4, color='C1', label='dN nominal (North)')
axs[1].axhline(0, color='black', lw=0.8)
axs[1].set_title(f'Position relative to the start (50%/95% bands, lowpass={CUTOFF_HZ} Hz)')
axs[1].set_xlabel('t (s)'); axs[1].set_ylabel('displacement (m)')
axs[1].legend(fontsize=8, ncol=2); axs[1].grid(alpha=0.3)

plt.tight_layout()
save_fig(fig, 'fig06_mc_bands',
         'Filtered plan-view trace and Monte Carlo uncertainty bands (50%/95%) of the relative position.')

# --- 3x1 figure: raw/filtered traces, |difference| histogram, signed difference trace ---
diff_E_cm_abs = np.abs(rel_serie_raw[:, 0] - rel_serie[:, 0]) * 100
diff_N_cm_abs = np.abs(rel_serie_raw[:, 1] - rel_serie[:, 1]) * 100
diff_E_cm_signed = (rel_serie_raw[:, 0] - rel_serie[:, 0]) * 100
diff_N_cm_signed = (rel_serie_raw[:, 1] - rel_serie[:, 1]) * 100

fig, axs = plt.subplots(3, 1, figsize=(9, 12))

# (1) raw vs. filtered traces
axs[0].plot(t_serie, rel_serie_raw[:, 0], 'o-', ms=3, lw=0.8, color='C0', alpha=0.5, label='dE raw (picked)')
axs[0].plot(t_serie, rel_serie[:, 0], '-', lw=2, color='C0', label=f'dE filtered ({CUTOFF_HZ} Hz)')
axs[0].plot(t_serie, rel_serie_raw[:, 1], 'o-', ms=3, lw=0.8, color='C1', alpha=0.5, label='dN raw (picked)')
axs[0].plot(t_serie, rel_serie[:, 1], '-', lw=2, color='C1', label=f'dN filtered ({CUTOFF_HZ} Hz)')
axs[0].axhline(0, color='black', lw=0.8)
axs[0].set_title('Components: raw vs. filtered')
axs[0].set_xlabel('t (s)'); axs[0].set_ylabel('displacement (m)')
axs[0].legend(fontsize=8); axs[0].grid(alpha=0.3)

# (2) |difference| histogram
bins = 30
axs[1].hist(diff_E_cm_abs, bins=bins, alpha=0.5, color='C0', label=f'|E raw - filtered| (mean={diff_E_cm_abs.mean():.2f} cm)')
axs[1].hist(diff_N_cm_abs, bins=bins, alpha=0.5, color='C1', label=f'|N raw - filtered| (mean={diff_N_cm_abs.mean():.2f} cm)')
axs[1].set_xlabel('|difference| (cm)')
axs[1].set_ylabel('frequency (n. of frames)')
axs[1].set_title(f'Distribution of the filter correction (lowpass={CUTOFF_HZ} Hz)')
axs[1].legend(fontsize=8)
axs[1].grid(alpha=0.3)

# (3) signed difference over time
axs[2].plot(t_serie, diff_E_cm_signed, '-o', ms=3, lw=1, color='C0', label='E raw - filtered')
axs[2].plot(t_serie, diff_N_cm_signed, '-o', ms=3, lw=1, color='C1', label='N raw - filtered')
axs[2].axhline(0, color='black', lw=0.8)
axs[2].set_xlabel('t (s)')
axs[2].set_ylabel('difference (cm)')
axs[2].set_title(f'Filter correction over time (lowpass={CUTOFF_HZ} Hz)')
axs[2].legend(fontsize=8)
axs[2].grid(alpha=0.3)
axs[2].set_ylim(-10, 10)

plt.tight_layout()
save_fig(fig, 'fig07_raw_vs_filtered',
         'Raw vs. filtered traces, absolute-difference histogram, and signed difference over time (Figures 7a-7c combined).')

# --- Per-frame position covariance (from the MC, already filtered) ---
# Each C_frames[i] is the 2x2 (E,N) covariance of the N_SIM simulations at frame i.
# Unlike the raw-vs-filtered residual, this is clean of the physical error sources
# (SIGMA_CLICK_PX, SIGMA_GCP_WORLD_M) and never touches the real signal -> no window to exclude.
C_frames = np.array([np.cov(sims_world[:, i, :].T) for i in range(n_pts)])   # (n_pts, 2, 2)

# --- Diagnostic: is the covariance reasonably stable over time? ---
eigvals = np.array([np.linalg.eigvalsh(C) for C in C_frames])              # (n_pts, 2) ascending
eigvecs = np.array([np.linalg.eigh(C)[1] for C in C_frames])               # (n_pts, 2, 2)
angles = np.degrees(np.arctan2(eigvecs[:, 1, 1], eigvecs[:, 0, 1])) % 180
aniso_ratio = np.sqrt(eigvals[:, 1] / np.clip(eigvals[:, 0], 1e-12, None))

fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
axs[0].plot(t_serie, np.sqrt(eigvals[:, 0]) * 100, label='minor sigma (cm)')
axs[0].plot(t_serie, np.sqrt(eigvals[:, 1]) * 100, label='major sigma (cm)')
axs[0].set_ylabel('sigma (cm)'); axs[0].legend(fontsize=8); axs[0].grid(alpha=0.3)
axs[0].set_title('Stability of the per-frame position covariance')

axs[1].plot(t_serie, aniso_ratio)
axs[1].axhline(aniso_ratio.mean(), color='k', ls='--', lw=1, label=f'mean={aniso_ratio.mean():.2f}')
axs[1].set_ylabel('anisotropy ratio'); axs[1].legend(fontsize=8); axs[1].grid(alpha=0.3)

axs[2].plot(t_serie, angles)
axs[2].set_ylabel('major-axis angle (deg)'); axs[2].set_xlabel('t (s)'); axs[2].grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, 'fig05b_covariance_stability',
         'Per-frame position covariance stability diagnostic: minor/major sigma, anisotropy ratio, principal-axis angle.')

cv_sigma_major = np.sqrt(eigvals[:, 1]).std() / np.sqrt(eigvals[:, 1]).mean()
print(f'CV of the major sigma over time = {cv_sigma_major*100:.1f}% '
      f'({"stable, averaging over time is reasonable" if cv_sigma_major < 0.15 else "varies a lot, use per-frame in the fusion"})')
print(f'Time-averaged covariance (m^2): {C_frames.mean(axis=0).ravel()}')

# --- Additional diagnostic: excluding the edges (worse homography conditioning there) ---
EDGE_TRIM = 4   # frames excluded on each end, for this diagnostic only
sigma_major = np.sqrt(eigvals[:, 1])
sigma_major_trimmed = sigma_major[EDGE_TRIM:-EDGE_TRIM]
cv_trimmed = sigma_major_trimmed.std() / sigma_major_trimmed.mean()
print(f'CV excluding {EDGE_TRIM} frames from each edge: {cv_trimmed*100:.1f}% '
      f'({"stable" if cv_trimmed < 0.15 else "still varies"})')

# --- Average covariance, computed over the interior stretch (excludes edges) ---
C_mean_trimmed = C_frames[EDGE_TRIM:-EDGE_TRIM].mean(axis=0)
print(f'Average covariance (interior stretch, m^2): {C_mean_trimmed.ravel()}')

# --- Save what's needed to fuse this view+target with the others later ---
out_path = os.path.join(OUTPUT_PATH, f'mc_result_{VIEW_NAME}_{TARGET}.npz')
np.savez(out_path,
         t_serie=t_serie,
         rel_serie=rel_serie,           # nominal filtered trace (relative to the start)
         rel_serie_raw=rel_serie_raw,   # nominal raw trace
         pts_world_serie=pts_world_serie,
         C_frames=C_frames,             # per-frame (E,N) covariance, from the MC
         cutoff_hz=CUTOFF_HZ,
         sigma_click_px=SIGMA_CLICK_PX,
         sigma_gcp_world_m=SIGMA_GCP_WORLD_M)
print(f'\nSaved: {out_path}')
